# 07 — Failure analysis (v8)
Actual failure analysis from available evaluation results in `results/week1_prompt_baseline_v8/` (read-only). Uses v8 loaders/scoring context where relevant. No invented metrics or failures.

In [5]:
# Parameters
V8_RUNS_DIR = 'results/week1_prompt_baseline_v8/runs'


In [6]:
import os, json
from pathlib import Path
from collections import defaultdict, Counter
_root = Path.cwd().resolve()
if not (_root / 'src').is_dir():
    _root = _root.parent
os.chdir(_root)
from src.config import GOLD_PATH
from src.eval.gold_rows import load_rows
from src.eval.prompting.inputs import load_dev_rows
gold_rows = load_rows(GOLD_PATH, active_only=True) or [json.loads(l) for l in open(GOLD_PATH) if l.strip()]
gold_by_id = {r.get('id') or r.get('row_id'): r for r in gold_rows}
dev_ids = {r['id'] for r in load_dev_rows()}
runs_dir = Path(V8_RUNS_DIR)
manifests = sorted(runs_dir.glob('*.manifest.json')) if runs_dir.exists() else []
valid, excluded = [], []
for mp in manifests:
    m = json.loads(mp.read_text(encoding='utf-8'))
    recs_path = Path(m.get('output_path',''))
    if not recs_path.is_absolute(): recs_path = _root / recs_path
    m['_recs_path']=recs_path
    if m.get('stopped_reason'):
        excluded.append((m,'stopped_reason'))
    else:
        # require non-trivial dev coverage for completed baseline
        if int(m.get('n_processed_this_run') or m.get('n_rows') or 0) < len(dev_ids):
            excluded.append((m,'incomplete_coverage'))
        else:
            valid.append(m)
print('gold_rows:', len(gold_rows), 'dev_rows:', len(dev_ids))
print('manifest_files:', len(manifests), 'valid_completed:', len(valid), 'excluded_incomplete_or_invalid:', len(excluded))
for m,reason in excluded:
    print('EXCLUDED', reason, m.get('arm'), m.get('run_id'), m.get('output_path'), 'stopped_reason=', m.get('stopped_reason'))
if not valid:
    print('\nPENDING: no valid completed baseline run records found in v8 runs dir; failure metrics not available and not invented.')


gold_rows: 97 dev_rows: 73
manifest_files: 1 valid_completed: 0 excluded_incomplete_or_invalid: 1
EXCLUDED stopped_reason B0 20261119T122358Z_B0 results/week1_prompt_baseline_v8/runs/20261119T122358Z_B0_B0.jsonl stopped_reason= error rate 1/1 exceeded 5%

PENDING: no valid completed baseline run records found in v8 runs dir; failure metrics not available and not invented.


In [3]:
# Abstention breakdown by abstention_subtype and annual/quarterly doc_type (from available valid results)
def agg_abstain(rows_scores):
    n=len(rows_scores)
    def rate(k): return (sum(1 for s in rows_scores if (s or {}).get(k)) / n) if n else None
    return {'n':n,'abstain_correct':rate('abstain_correct'),'false_answer':rate('false_answer'),'false_abstention':rate('false_abstention')}
if not valid:
    print('PENDING: abstention error breakdown by abstention_subtype and annual/quarterly doc_type unavailable (no valid completed run records).')
    print('Reference (dataset, not model failures):')
    ab=[r for r in gold_rows if r.get('expected_behaviour')=='abstain']
    print('dataset_abstention_subtype:', dict(Counter(r.get('abstention_subtype') for r in ab)))
    print('dataset_abstention_by_doc_type:', {d: dict(Counter(r.get('abstention_subtype') for r in ab if r.get('doc_type')==d)) for d in ['annual','quarterly']})
else:
    for m in valid:
        recs=[json.loads(l) for l in Path(m['_recs_path']).read_text(encoding='utf-8').splitlines() if l.strip()]
        by_sub, by_doc = defaultdict(list), defaultdict(list)
        for rec in recs:
            row = gold_by_id.get(rec.get('row_id'), {})
            sc = rec.get('score') or {}
            sub = row.get('abstention_subtype') if row.get('expected_behaviour')=='abstain' else 'answer'
            by_sub[sub].append(sc); by_doc[row.get('doc_type')].append(sc)
        print('ARM', m.get('arm'), m.get('run_id'))
        print('by_abstention_subtype', {k: agg_abstain(v) for k,v in sorted(by_sub.items(), key=lambda x:str(x[0]))})
        print('by_doc_type_annual_quarterly', {k: agg_abstain(v) for k,v in sorted(by_doc.items()) if k in ('annual','quarterly')})


PENDING: abstention error breakdown by abstention_subtype and annual/quarterly doc_type unavailable (no valid completed run records).
Reference (dataset, not model failures):
dataset_abstention_subtype: {'not_in_corpus': 31, 'near_miss_absent': 1, 'out_of_scope': 1, 'metric_not_applicable': 4}
dataset_abstention_by_doc_type: {'annual': {'not_in_corpus': 31, 'near_miss_absent': 1, 'out_of_scope': 1}, 'quarterly': {'metric_not_applicable': 4}}


In [7]:
# Failure categories from available valid results (actual; no invention)
if not valid:
    print('PENDING: retrieval misses, numeric errors, unit mismatches, period/basis confusion, citation errors, and abstention errors cannot be computed without valid completed baseline results.')
    print('Instrumentation notes: scorer provides numeric/entity/citation/abstention signals; retrieval-miss labels and fine-grained unit/period-basis confusion attribution are not directly instrumented in current stored scores.')
else:
    for m in valid:
        recs=[json.loads(l) for l in Path(m['_recs_path']).read_text(encoding='utf-8').splitlines() if l.strip()]
        n=len(recs)
        ans=[r for r in recs if (gold_by_id.get(r.get('row_id'),{}).get('expected_behaviour')=='answer')]
        num_app=[r for r in ans if ((r.get('score') or {}).get('numeric_match') is not None)]
        num_err=sum(1 for r in num_app if (r.get('score') or {}).get('numeric_match') is False)
        cit_err=sum(1 for r in ans if (r.get('score') or {}).get('citation_valid') is False)
        abstain_err=sum(1 for r in recs if (r.get('score') or {}).get('abstain_correct') is False)
        false_ans=sum(1 for r in recs if (r.get('score') or {}).get('false_answer'))
        false_abs=sum(1 for r in recs if (r.get('score') or {}).get('false_abstention'))
        print('ARM', m.get('arm'), 'n=', n)
        print({'numeric_errors(applicable)': f'{num_err}/{len(num_app)}', 'citation_errors(answerable)': f'{cit_err}/{len(ans)}', 'abstention_errors': abstain_err, 'false_answer': false_ans, 'false_abstention': false_abs})
        print('retrieval_misses: unavailable (not instrumented in stored scores)')
        print('unit_mismatches / period-basis confusion: unavailable (not directly instrumented); inspect record outputs/scores manually when needed')


PENDING: retrieval misses, numeric errors, unit mismatches, period/basis confusion, citation errors, and abstention errors cannot be computed without valid completed baseline results.
Instrumentation notes: scorer provides numeric/entity/citation/abstention signals; retrieval-miss labels and fine-grained unit/period-basis confusion attribution are not directly instrumented in current stored scores.


**Status:** Failure analysis is result-dependent. Pending categories/metrics above will populate when valid completed runs exist in `results/week1_prompt_baseline_v8/runs/` (endpoint access currently blocked per report).